In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(r"D:\OneDrive - Coforge Limited\Documents\CAPSTONE PROJECT 1\Section 2 - Loan_Default_FastAPI_App\data\Loan_Default.csv")
df = df.drop_duplicates()

# Target
target = "Status"
X = df.drop(columns=[target])
y = df[target]

# OPTIONAL: if you want to REMOVE dtir1 permanently, drop it here (and never send it in API)
# X = X.drop(columns=["dtir1"])

# -----------------------------
# Split FIRST (no leakage)
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# -----------------------------
# Column lists from RAW dataframe
# -----------------------------
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

# -----------------------------
# Preprocess
# -----------------------------
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# robust for unseen categories at inference
try:
    ohe = OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", drop=None, sparse=False)

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe)
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

# -----------------------------
# FULL training pipeline: preprocess -> SMOTE -> KNN
# -----------------------------
pipe = ImbPipeline(steps=[
    ("preprocess", preprocess),
    ("smote", SMOTE(random_state=42)),
    ("model", KNeighborsClassifier(n_neighbors=7))
])

# Fit
pipe.fit(X_train, y_train)

# Evaluate
pred = pipe.predict(X_test)
print("Test F1:", f1_score(y_test, pred))

# Feature names (optional, but helpful for debugging)
try:
    feature_names = pipe.named_steps["preprocess"].get_feature_names_out().tolist()
except Exception:
    feature_names = None

# Save bundle (so inference uses RAW columns)
bundle = {
    "model": pipe,                         # ✅ FULL pipeline
    "features_raw": X_train.columns.tolist(),  # ✅ RAW CSV columns
    "features_transformed": feature_names,     # optional
    "target": target
}

joblib.dump(bundle, r"D:\OneDrive - Coforge Limited\Documents\CAPSTONE PROJECT 1\Section 2 - Loan_Default_FastAPI_App\model\loan_default_bundle.pkl")
print("✅ Saved: loan_default_bundle.pkl")

KeyboardInterrupt: 